In [1]:
import numpy as np
import matplotlib.pyplot as plt
#write a code to 

In [2]:
import os
import numpy as np

dir = ['./generative_results',\
       ]
# Find all .npy files in the directories that contain dicts
# ignore the files that start with 'sequences_'
# do not plot ourGREEDY results
npy_files = []
method_names = []
for d in dir:
    for file in os.listdir(d):
        if file.endswith('.npy') and not file.startswith(('sequences_', 'gflownet','cbas', 'dbas', 'bo','dynappo')) and 'ourGREEDY' not in file:
            npy_files.append(os.path.join(d, file))
            method_names.append(os.path.basename(file).split('_')[0])

# reshuffle to make "Ours" last if present
if 'Ours' in method_names:
    idx = method_names.index('Ours')
    npy_files.append(npy_files.pop(idx))
    method_names.append(method_names.pop(idx))
print(npy_files)
print(method_names)

# in the code below, replace f'File {i+1}' with method_names[i]

# Load all dicts from the found .npy files
dicts = []
for file in npy_files:
    arr = np.load(file, allow_pickle=True)
    if isinstance(arr.item(), dict):
        dicts.append(arr.item())

common_keys = set.intersection(*(set(d.keys()) for d in dicts))

# Dictionary to store vals for each method and key
method_vals_storage = {key: {} for key in common_keys}
key_to_minmax = {}
for key in common_keys:
    max_values = []  # Store max value for each method
    
    for i, dct in enumerate(dicts):
        vals = np.array(dct[key])
        
        if len(vals.shape) == 3:
            vals = vals[:, 0:10, 2]
            vals = np.mean(vals, axis=0)
        else:
            vals = np.mean(vals, axis=0)
        
        # Store vals for this method
        method_vals_storage[key][method_names[i]] = vals
        
        # Get max value for this method
        max_val = np.max(vals)
        max_values.append(max_val)
    
    # Find minimum of the max values
    min_max_value = np.min(max_values)
    min_max_method_idx = np.argmin(max_values)
    min_max_method = method_names[min_max_method_idx]
    key_to_minmax[key] = min_max_value ## this is what we need 
    
print(key_to_minmax)

['./generative_results/our_iteration_times_1.npy', './generative_results/our_1.npy']
['our', 'our']
{50: 3.4927378967909, 100: 3.772722669085635, 20: 2.3929566040100245}


## Preparing the data for metric calculation

In [3]:
data_path = "/home/apa2237/generative_model_work/datasets/acp_gravy"

top_per = 40

seq_start = np.load(f'{data_path}/seq_test.npy', allow_pickle=True).reshape((-1,1))
y_start = np.load(f'{data_path}/y_test.npy', allow_pickle=True).reshape((-1,1))

print('Before filtering',len(seq_start), len(y_start))
print('Mean before', np.mean(y_start))
print('Min before', np.min(y_start))
print('Max before', np.max(y_start))

cap = max(y_start)
floor = 0.1#min(y_start)
cutoff = (floor)*(100+top_per)/100
print('Cutoff', cutoff)
below_idx = (y_start<cutoff)
# print('Below_idx', np.sum(below_idx*1))
print(below_idx.shape,seq_start.shape)

seq_start = seq_start[below_idx]
y_start = y_start[below_idx]

print('Filtering==================')
print('After filtering',len(seq_start), len(y_start))
print('Mean after', np.mean(y_start))
print('Max after', np.max(y_start))

# print(seq_start)


Before filtering 150 150
Mean before -0.2913379289536502
Min before -2.630434782608696
Max before 1.6238095238095238
Cutoff 0.14
(150, 1) (150, 1)
Filtering==================
After filtering 105 105
Mean after -0.713256501147901
Max after 0.13478260869565215


In [4]:
from polyleven import levenshtein

def edit_dist(seq1, seq2):
    return levenshtein(seq1, seq2) / 1

import itertools

def mean_pairwise_distances(args, seqs):
    dists = []
    for pair in itertools.combinations(seqs, 2):
        dists.append(edit_dist(*pair))
    return np.mean(dists)

def novelty(list_1, list_2):
    dists = []
    for s1, s2 in itertools.product(list_1, list_2):  # cross pairs only
        # print(s1,s2)
        dists.append(edit_dist(s1, s2))
    return np.mean(dists) if dists else 0.0


In [5]:
def auc_cal(prop):
    for k in prop.keys():  # show up to 3 keys
        if k not in [20, 50, 100]:
            continue
        # Get data
        if len(prop[k].shape) == 2:
            temp = prop[k]
        else:
            temp = prop[k][..., 2]
        area = []
        for j in range(temp.shape[0]):
            y = temp[j,:]
            # shifting y so that min is zero
            y = y - np.min(y)
            x = np.arange(len(y))
            auc = np.trapz(y, x)
            area.append(auc)
        print(f"Samples:{k}, AUC:{np.mean(area):.2f} ± {np.std(area):.2f}")
    # return area

In [6]:
def find_nearest(arr, target):
    L,R = 0, len(arr)-1
    while L<R:
        mid = (L+R)//2
        if arr[mid]<target:
            L = mid+1
        else:
            R = mid
        
    return R

In [7]:
def cal_d_n(sequences,prop_ada):
    div_res = []
    for k1 in sequences.keys():
        if k1 not in [20, 50, 100]:
            continue
        sum_dist = []
        novel_dist = []
        for k2 in sequences[k1].keys():
            if len(prop_ada[k1][k2].shape)==2:
                idx = find_nearest(prop_ada[k1][k2][1:,2], key_to_minmax[k1])
            else:
                idx = find_nearest(prop_ada[k1][k2][1:], key_to_minmax[k1])
            

            # # if sequences[k1][k2][idx]>=key_to_minmax[k1]:
            # if len(prop_ada[k1][k2].shape)==2:
            #     if 0.8*key_to_minmax[k1] <= prop_ada[k1][k2][idx,2] <= 1.2*key_to_minmax[k1]:
            #         sim = mean_pairwise_distances(None, sequences[k1][k2][idx+1])
            #         sum_dist.append(sim)
            # else:
            #     if 0.8*key_to_minmax[k1] <= prop_ada[k1][k2][idx] <= 1.2*key_to_minmax[k1]:
            sim = mean_pairwise_distances(None, sequences[k1][k2][idx+1])
            sum_dist.append(sim)
            
            nov = novelty(sequences[k1][k2][9],seq_start)
            novel_dist.append(nov)
            
        print(f"Samples:{k1}, Diversity:{np.mean(sum_dist):.2f} ± {np.std(sum_dist):.2f}, \
        Novelty:{np.mean(novel_dist):.2f} ± {np.std(novel_dist):.2f}")
        div_res.append(np.mean(sum_dist))
        
    return div_res


## AUC

In [ ]:
data_dir = './generative_results'


prop_mode = np.load(f'{data_dir}/our_1.npy', allow_pickle=True).tolist()
sequences_mode = np.load(f'{data_dir}/sequences_our_1.npy', allow_pickle=True).tolist()
print('--------------IDEAS----------------')
auc_cal(prop_mode)


--------------IDEAS----------------
Samples:20, AUC:20.86 ± 2.99
Samples:50, AUC:34.73 ± 2.87
Samples:100, AUC:39.78 ± 1.49


## Diversity and Novelty

In [9]:

print('--------------IDEAS----------------')
div_idea = cal_d_n(sequences_mode, prop_mode  )


--------------IDEAS----------------
Samples:20, Diversity:4.76 ± 1.30,         Novelty:26.90 ± 0.40
Samples:50, Diversity:9.22 ± 2.80,         Novelty:28.21 ± 1.73
Samples:100, Diversity:11.13 ± 1.20,         Novelty:27.89 ± 0.67
